In [6]:
import cv2

In [7]:
import os
import time
import numpy as np
import torch
from PIL import Image
from transformers import pipeline

In [8]:
device = 0 if torch.cuda.is_available() else -1
depth = pipeline("depth-estimation",
                 model="depth-anything/Depth-Anything-V2-Small-hf",
                 device=device)


Loading weights: 100%|██████████| 287/287 [00:00<00:00, 2084.15it/s]


In [9]:
RGB_DIR = "data/rgb"
DEPTH_DIR = "data/depth"
os.makedirs(RGB_DIR, exist_ok=True)
os.makedirs(DEPTH_DIR, exist_ok=True)

cap = cv2.VideoCapture(0)

prev_t = time.time()
fps = 0.0
i = 0

while True: 
    ok, frame = cap.read()
    if not ok:
        break

    now = time.time()
    fps = 1 / (now - prev_t) if now > prev_t else fps
    prev_t = now

    rgb = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    depth_raw = np.array(depth(rgb)["depth"], dtype=np.float32)  # raw depth, for training target

    # save the pair: original frame + raw (unnormalized) depth map
    cv2.imwrite(f"{RGB_DIR}/{i:06d}.png", frame)
    np.save(f"{DEPTH_DIR}/{i:06d}.npy", depth_raw)
    i += 1

    d = cv2.normalize(depth_raw, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)  # for display only

    # d = cv2.applyColorMap(d, cv2.COLORMAP_INFERNO) # rgb
    d = cv2.cvtColor(d, cv2.COLOR_GRAY2BGR) # grayscale

    d = cv2.resize(d, (frame.shape[1], frame.shape[0]))

    combined = np.hstack([frame, d])
    cv2.putText(combined, f"FPS: {fps:.1f}  saved: {i}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Webcam | Depth", combined)

    if cv2.waitKey(1) & 0xFF == ord('q'):  # q to quit
            break

cap.release()
cv2.destroyAllWindows()

In [17]:
import sys
import threading
sys.path.append("student")
from model import StudentDepthNet

STUDENT_SIZE = (192, 144)  # (width, height), must match training img_size
PANEL_W = 400  # shrink each displayed panel so all 3 fit on screen side by side

student = StudentDepthNet()
student.load_state_dict(torch.load("student/checkpoints/best.pt", map_location="cpu"))
student.eval()

# Teacher runs on its own background thread so its (slow) inference never
# blocks the webcam/student loop -- the main loop just reads whatever the
# teacher last finished, instead of waiting for it every frame.
teacher_lock = threading.Lock()
teacher_state = {"frame": None, "disp": None, "ms": 0.0, "age": 0.0}
stop_event = threading.Event()

def teacher_worker():
    while not stop_event.is_set():
        with teacher_lock:
            frame_in = teacher_state["frame"]
        if frame_in is None:
            time.sleep(0.005)
            continue

        t0 = time.time()
        rgb = Image.fromarray(cv2.cvtColor(frame_in, cv2.COLOR_BGR2RGB))
        teacher_raw = np.array(depth(rgb)["depth"], dtype=np.float32)
        disp = cv2.normalize(teacher_raw, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        disp = cv2.cvtColor(disp, cv2.COLOR_GRAY2BGR)
        ms = (time.time() - t0) * 1000

        with teacher_lock:
            teacher_state["disp"] = disp
            teacher_state["ms"] = ms
            teacher_state["done_t"] = time.time()
            # only clear "frame" once consumed, so we don't redo the same frame
            teacher_state["frame"] = None

thread = threading.Thread(target=teacher_worker, daemon=True)
thread.start()

cap = cv2.VideoCapture(0)
prev_t = time.time()
fps = 0.0

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        now = time.time()
        fps = 1 / (now - prev_t) if now > prev_t else fps
        prev_t = now

        # hand the teacher the freshest frame whenever it's free (non-blocking)
        with teacher_lock:
            if teacher_state["frame"] is None:
                teacher_state["frame"] = frame.copy()
            teacher_disp = teacher_state["disp"]
            t_teacher = teacher_state["ms"]
            teacher_age = time.time() - teacher_state.get("done_t", now)

        t0 = time.time()
        small = cv2.resize(frame, STUDENT_SIZE, interpolation=cv2.INTER_AREA)
        small_rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        inp = torch.from_numpy(small_rgb).permute(2, 0, 1).unsqueeze(0)
        with torch.no_grad():
            student_raw = student(inp)[0, 0].numpy()
        t_student = time.time() - t0

        student_disp = (np.clip(student_raw, 0, 1) * 255).astype(np.uint8)
        student_disp = cv2.cvtColor(student_disp, cv2.COLOR_GRAY2BGR)

        panel_h = int(frame.shape[0] * PANEL_W / frame.shape[1])
        frame_small = cv2.resize(frame, (PANEL_W, panel_h))
        student_small = cv2.resize(student_disp, (PANEL_W, panel_h))
        if teacher_disp is not None:
            teacher_small = cv2.resize(teacher_disp, (PANEL_W, panel_h))
        else:
            teacher_small = np.zeros_like(frame_small)

        combined = np.hstack([frame_small, teacher_small, student_small])
        cv2.putText(combined, f"FPS {fps:.1f}  teacher {t_teacher:.0f}ms (age {teacher_age*1000:.0f}ms)  student {t_student*1000:.0f}ms",
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.putText(combined, "Webcam", (10, panel_h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.putText(combined, "Teacher", (PANEL_W + 10, panel_h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.putText(combined, "Student", (2 * PANEL_W + 10, panel_h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        cv2.imshow("Webcam | Teacher | Student", combined)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    stop_event.set()
    thread.join(timeout=2)
    cap.release()
    cv2.destroyAllWindows()